# PHerc0846A Inference

Forward-only inference with a flat depth-13 surface. `DEPTH_OFFSET = -1` selects slices 9-16; change the offset immediately below `CHECKPOINT` in the settings cell. Saved checkpoint geometry and normalization are replayed, while this offset explicitly overrides saved offsets.

Run the setup cells, then any individual segment cell. The final cell processes the remaining PHerc0846A quick patches into one `quick_patches` subfolder, excluding segments with individual cells. Figures contain prediction and the composite of the selected input slices. Volumes and masks must already be assembled; this notebook does not reverse volumes, detect surfaces, or modify labels.

In [ ]:
# [] = one depth; [low, high] = ensemble every offset low..high around the usual depth ([-3, 3] -> 7 depths)
depth_ensemble = [-2,2]
# how ensemble depths combine: 'z-score' (top-2), 'max', 'mid-boost' (k=3), 'mean' (top-2); '' = 'mid-boost'
ensemble_method = 'mid-boost'

CHECKPOINT = "/vesuvius/models/runs/nop29_resume/model_epoch_6.pth"
DEPTH_OFFSET = -1

FLAT_DEPTH = 13
EVAL_STRIDE = 32
PREVIEW_DOWNSCALE = 8 
DEVICE = "auto"

import os
import sys
from pathlib import Path

REPO = Path("/vesuvius")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.chdir(REPO)


In [ ]:
import gc
import json
import tempfile
import types

import cv2
import numpy as np
import torch
import zarr
from IPython.display import display
from PIL import Image

from utils.config import Config, ScrollConfig, resolve_run_config
from utils.fast_figure import colorize, gray_rgb, save_panel_grid
from utils.model import create_model
from utils.norm import (
    ANCHOR_GAP_LEVEL, ANCHOR_PAPYRUS_LEVEL, UNIFIED_CACHE_PATH,
    compute_norm, compute_surface_anchors, load_cached_norm,
)
from utils.platform import get_zarr_dir
from utils.visualizer import TensorboardVisualizer, predict_tiles

Image.MAX_IMAGE_PIXELS = None
MODEL_PATH = Path(CHECKPOINT).resolve()
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"Checkpoint is missing: {MODEL_PATH}")
RUN_CONFIG, RUN_CONFIG_PATH = resolve_run_config(MODEL_PATH)
C = Config()
for section in ("data", "model"):
    for name, value in RUN_CONFIG[section].items():
        if section == "data" and name == "scrolls":
            C.data.scrolls = [ScrollConfig(**item) for item in value]
        elif name != "probe_rois" and hasattr(getattr(C, section), name):
            setattr(getattr(C, section), name, value)
C.model.compile_model = False
C.tra.supcon = False
C.tra.dann = False
C.device = ("cuda" if torch.cuda.is_available() else "cpu") if DEVICE == "auto" else DEVICE
C.data.eval_stride = EVAL_STRIDE
C.data.surface_window_offset = int(DEPTH_OFFSET)
C.data.surface_window_offset_by_scroll = {}
ZARR_DIR = Path(get_zarr_dir())
C.data.zarr_path = str(ZARR_DIR)
DEPTH = int(C.data.depth)
if DEPTH != 8 or not C.model.surface_teacher_input or not C.data.surface_relative_depth_window:
    raise ValueError("Select a literal-surface, eight-slice checkpoint")
if EVAL_STRIDE <= 0 or EVAL_STRIDE > C.model.multitile_grid * C.model.multitile_subtile:
    raise ValueError("Inference stride must be positive and no larger than the prediction center")
WINDOW_START = FLAT_DEPTH - (DEPTH - 1) // 2 + int(DEPTH_OFFSET)
WINDOW_END = WINDOW_START + DEPTH
NORM_MODE = str(C.data.norm_mode)
MODEL_NAME = f"{MODEL_PATH.parent.parent.name}_{MODEL_PATH.parent.name}_{MODEL_PATH.stem}"
OUT_DIR = REPO / "output" / "0846A" / MODEL_NAME / f"offset_{DEPTH_OFFSET:+d}"
CACHE_ROOT = REPO / "_ves_tmp" / "0846A_inference"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

# every patch is read with a flat depth-FLAT_DEPTH surface (load_inputs); detected surfaces are never used
MAIN_PATCHES = {
    "auto_grown_20260927134928530_abf": {"zid": "20260927134928", "norm_id": 20260927134931},
    "auto_grown_20261012065803387_straight_abf": {"zid": "20261012065810", "norm_id": 20261012065811},
    "auto_grown_20261012065803387_straight_abf_m28": {"zid": "20261012065812", "norm_id": 20261012065813},
    "auto_grown_20261012073244544_abf": {"zid": "20261012073244", "norm_id": 20261012073246},
    "auto_grown_20261012073244544_abf_p29": {"zid": "20261012073249", "norm_id": 20261012073250},
    "auto_grown_20261012081237255_abf": {"zid": "20261012081237", "norm_id": 20261012081238},
}
for patch in MAIN_PATCHES.values():
    patch["mask_dir"] = REPO / "masks"
QUICK_ROOT = REPO / "tifxyz" / "quick_patches_from_D" / "PHerc0846A"
if not QUICK_ROOT.is_dir():
    raise FileNotFoundError(QUICK_ROOT)
QUICK_PATCHES = {
    path.name: {"zid": path.name, "mask_dir": REPO / "masks" / "extras"}
    for path in sorted(QUICK_ROOT.iterdir()) if path.is_dir() and (path / "x.tif").is_file()
}
EXTRA_INDIVIDUAL_NAMES = (
    "auto_grown_20260925180043604_abf", "auto_grown_20260927214603141_abf",
    "auto_grown_20260926143326538_abf", "auto_grown_20261002131432623_abf",
)
for name in EXTRA_INDIVIDUAL_NAMES:
    if name not in QUICK_PATCHES:
        raise FileNotFoundError(f"Individual quick-patch mesh is missing: {name}")
INDIVIDUAL_NAMES = (*MAIN_PATCHES, *EXTRA_INDIVIDUAL_NAMES)
PATCHES = {**QUICK_PATCHES, **MAIN_PATCHES}
REMAINING_QUICK_NAMES = tuple(name for name in QUICK_PATCHES if name not in INDIVIDUAL_NAMES)
MODEL = None


def supervision_role(name):
    patch = PATCHES[name]
    identities = {name, str(patch["zid"]), str(patch.get("norm_id", ""))}
    domains = C.data.train_scroll_dict or {}
    excluded = set(C.data.holdout_domains or [])
    trained = {str(sid) for domain, ids in domains.items() if domain not in excluded for sid in ids}
    if not domains:
        trained = {str(scroll.scroll_id) for scroll in C.data.scrolls}
    return "TRAIN (subject to saved label split)" if identities & trained else "VIS-ONLY (not supervised by this checkpoint)"


print(f"Model: {MODEL_PATH}\nConfig: {RUN_CONFIG_PATH}\nForward slices: {WINDOW_START}-{WINDOW_END - 1} (flat depth {FLAT_DEPTH})")
print(f"Output: {OUT_DIR}\nIndividual segments: {len(INDIVIDUAL_NAMES)}; remaining quick patches: {len(REMAINING_QUICK_NAMES)}")


In [ ]:
def get_model():
    global MODEL
    if MODEL is None:
        model, parameter_count = create_model(C)
        state = torch.load(MODEL_PATH, map_location="cpu", weights_only=True)
        if "model_state_dict" in state:
            state = state["model_state_dict"]
        elif "state_dict" in state:
            state = state["state_dict"]
        state = {name.removeprefix("module.").removeprefix("_orig_mod."): value for name, value in state.items()}
        state = {name: value for name, value in state.items() if not name.startswith(("supcon_head.", "domain_head."))}
        model.load_state_dict(state, strict=True)
        model.eval()
        MODEL = model
        print(f"Loaded {parameter_count:,} parameters on {C.device}")
    return MODEL


def input_paths(name):
    patch = PATCHES[name]
    volume_path = ZARR_DIR / f"{patch['zid']}.zarr"
    mask_path = patch["mask_dir"] / f"{patch['zid']}.png"
    missing = [str(path) for path in (volume_path, mask_path) if not path.exists()]
    if missing:
        raise FileNotFoundError("Assemble these inputs before inference: " + ", ".join(missing))
    return volume_path, mask_path


def flat_norm(patch, volume_path, mask_path, surface_depth, surface_confidence):
    if "norm_id" in patch and FLAT_DEPTH == 13:
        try:
            cached = load_cached_norm(patch["norm_id"], UNIFIED_CACHE_PATH, mode=NORM_MODE)
        except RuntimeError:
            cached = None
        if cached is not None:
            return cached
    if NORM_MODE == "raw255":
        return 0.0, 1.0, 0.0, 255.0
    private_cache = str(CACHE_ROOT / "global_norm.json")
    if NORM_MODE == "global":
        cached = load_cached_norm(patch["zid"], UNIFIED_CACHE_PATH, mode="global")
        cached = cached or load_cached_norm(patch["zid"], private_cache, mode="global")
        return cached or compute_norm(patch["zid"], str(ZARR_DIR), private_cache, mask_dir=str(patch["mask_dir"]))
    if NORM_MODE not in ("surface_anchor", "surface_anchor_knee"):
        raise ValueError(f"Unsupported normalization: {NORM_MODE}")
    attrs_path = volume_path / ".zattrs"
    signature = {
        "volume": str(volume_path.resolve()), "flat_depth": FLAT_DEPTH,
        "attrs_mtime": attrs_path.stat().st_mtime_ns if attrs_path.exists() else volume_path.stat().st_mtime_ns,
        "mask_mtime": mask_path.stat().st_mtime_ns, "shape": list(surface_depth.shape),
    }
    cache_path = CACHE_ROOT / f"{patch['zid']}_flat_anchors.json"
    if cache_path.is_file():
        cached = json.loads(cache_path.read_text())
        if cached.get("signature") == signature:
            return tuple(cached["norm"])
    with tempfile.TemporaryDirectory(dir=CACHE_ROOT) as temporary:
        surface_dir = Path(temporary) / str(patch["zid"])
        surface_dir.mkdir()
        np.save(surface_dir / "depth.npy", surface_depth)
        np.save(surface_dir / "confidence.npy", surface_confidence)
        anchors = compute_surface_anchors(patch["zid"], str(ZARR_DIR), surface_dir=temporary,
                                          mask_dir=str(patch["mask_dir"]))
    span = (anchors["papyrus"] - anchors["gap"]) / (ANCHOR_PAPYRUS_LEVEL - ANCHOR_GAP_LEVEL)
    lower = anchors["gap"] - ANCHOR_GAP_LEVEL * span
    norm = (0.0, 1.0, lower, lower + span)
    cache_path.write_text(json.dumps({"signature": signature, "norm": norm, "anchors": anchors}, indent=2))
    return norm


# most recent patch's inputs, so consecutive cells on one patch skip the reload
_INPUT_CACHE = {}


def load_inputs(name):
    import time

    volume_path, mask_path = input_paths(name)
    key = (name, (volume_path / ".zarray").stat().st_mtime_ns, mask_path.stat().st_mtime_ns)
    if key in _INPUT_CACHE:
        return _INPUT_CACHE[key]
    _INPUT_CACHE.clear()
    gc.collect()
    source = zarr.open(str(volume_path), mode="r")
    if source.attrs.get("layers_reversed_from") or source.attrs.get("facing_reversed"):
        raise ValueError(f"Forward inference requires the original volume: {volume_path}")
    with Image.open(mask_path) as image:
        mask = np.asarray(image.convert("L")) > 0
    height, width = mask.shape
    if tuple(source.shape[1:]) != (height, width):
        raise ValueError(f"Mask and volume frames differ for {name}: {mask.shape} vs {source.shape}")
    if not mask.any() or WINDOW_START < 0 or WINDOW_END > source.shape[0]:
        raise ValueError(f"Invalid mask or depth window for {name}")
    started = time.time()
    # small zarr chunks make the overlapping context-strip reads decompress each chunk many times
    volume = np.ascontiguousarray(source[:])
    volume.setflags(write=False)
    print(f"[preload] {name}: {volume.shape} {volume.nbytes / 1024**3:.2f} GiB in RAM in {time.time() - started:.1f}s")
    surface_depth = np.where(mask, FLAT_DEPTH, 255).astype(np.uint8)
    surface_confidence = np.where(mask, 255, 0).astype(np.uint8)
    coords = TensorboardVisualizer._gen_tile_coords(
        types.SimpleNamespace(c=C), (C.data.d_start, C.data.d_end),
        (0, height), (0, width), mask, z_step=DEPTH,
    )
    if not coords or any(coordinate[0] != 0 for coordinate in coords):
        raise ValueError(f"Expected one nonempty surface-relative pass for {name}")
    norm = flat_norm(PATCHES[name], volume_path, mask_path, surface_depth, surface_confidence)
    _INPUT_CACHE[key] = {"volume": volume, "mask": mask, "surface_depth": surface_depth,
                         "surface_confidence": surface_confidence, "coords": coords, "norm": norm}
    return _INPUT_CACHE[key]


In [ ]:
import importlib
import utils.fast_figure as figure_utils
import utils.depth_ensemble as depth_ensemble_utils

importlib.reload(figure_utils)
importlib.reload(depth_ensemble_utils)
depth_ensemble_utils.resolve(depth_ensemble, ensemble_method)


def render(name, output_subdir=None, preview=True):
    from utils.fast_figure import orient_display

    inputs = load_inputs(name)
    model = get_model()
    print(f"{name}: {supervision_role(name)}")
    height, width = inputs["mask"].shape
    offsets, method = depth_ensemble_utils.resolve(depth_ensemble, ensemble_method)
    prediction = depth_ensemble_utils.ensemble_predict(predict_tiles, offsets, method)(
        C, model, inputs["volume"], inputs["mask"], inputs["coords"],
        (0, height), (0, width), C.data.d_start, name, *inputs["norm"],
        surface_depth_map=inputs["surface_depth"],
        surface_confidence_map=inputs["surface_confidence"],
        surface_depth_offset=int(DEPTH_OFFSET),
    )
    ensemble_label = depth_ensemble_utils.describe([int(DEPTH_OFFSET) + o for o in offsets], method)
    projection = np.asarray(inputs["volume"][WINDOW_START:WINDOW_END]).max(axis=0)
    composite = np.clip(projection, 0, 255).astype(np.uint8) * inputs["mask"]
    display_prediction = orient_display(prediction, name)
    display_composite = orient_display(composite, name)
    panels = [[
        (f"{name} | {ensemble_label or f'forward {WINDOW_START}-{WINDOW_END - 1}'}",
         colorize(display_prediction, "gray_r", (0.78, 0.86, 0.95)), cv2.INTER_NEAREST, None),
        (f"composite {WINDOW_START}-{WINDOW_END - 1}", gray_rgb(display_composite), cv2.INTER_AREA, 9.362),
    ]]
    folder = OUT_DIR if output_subdir is None else OUT_DIR / output_subdir
    folder.mkdir(parents=True, exist_ok=True)
    output_path = folder / f"{name}.png"
    save_panel_grid(panels, display_composite.shape, str(output_path), gap_px=24)
    if preview:
        with Image.open(output_path) as saved:
            reduced = saved.resize((max(1, saved.width // PREVIEW_DOWNSCALE),
                                    max(1, saved.height // PREVIEW_DOWNSCALE)), Image.Resampling.LANCZOS)
        display(reduced)
    print(f"Saved {output_path}")
    del inputs, prediction, projection, composite, display_prediction, display_composite, panels
    gc.collect()
    if C.device == "cuda":
        torch.cuda.empty_cache()
    return output_path


def render_remaining_quick_patches():
    for name in REMAINING_QUICK_NAMES:
        input_paths(name)
    failures = {}
    outputs = []
    for index, name in enumerate(REMAINING_QUICK_NAMES, start=1):
        print(f"[{index}/{len(REMAINING_QUICK_NAMES)}] {name}")
        try:
            outputs.append(render(name, output_subdir="quick_patches", preview=False))
        except Exception as error:
            failures[name] = str(error)
            print(f"FAILED {name}: {error}")
    print(f"Saved {len(outputs)} quick-patch figures to {OUT_DIR / 'quick_patches'}")
    if failures:
        raise RuntimeError(f"Quick-patch failures: {failures}")
    return outputs

In [ ]:
# HELD OUT WINNING SHEET
render("auto_grown_20261012065803387_straight_abf")

In [ ]:
# HELD OUT WINNING SHEET, OFFSET +29 (-29 in VC3D)
render("auto_grown_20261012065803387_straight_abf_m28")

In [ ]:
# TRAIN in the all-patch baseline; the selected checkpoint's actual role is printed.
render("auto_grown_20260927134928530_abf")

In [ ]:
# TRAIN in the all-patch baseline; the selected checkpoint's actual role is printed.
render("auto_grown_20261012073244544_abf")

In [ ]:
# VIS-ONLY: not supervised by any configured run.
render("auto_grown_20260925180043604_abf")


In [ ]:
# VIS-ONLY: not supervised by any configured run.
render("auto_grown_20260927214603141_abf")


In [ ]:
# VIS-ONLY: not supervised by any configured run.
render("auto_grown_20260926143326538_abf")


In [ ]:
# VIS-ONLY: not supervised by any configured run.
render("auto_grown_20261002131432623_abf")


In [ ]:
# VIS-ONLY: remaining quick patches; individual entries above are excluded.
quick_patch_outputs = render_remaining_quick_patches()

In [ ]:
# OFFSET SWEEP, straight_abf: save offsets -4 through +4 (figures and raw .npy predictions) without displaying them.
STRAIGHT = "auto_grown_20261012065803387_straight_abf"
STRAIGHT_M28 = "auto_grown_20261012065803387_straight_abf_m28"


def sweep_offset_depth(patch_name):
    from utils.fast_figure import orient_display

    inputs = load_inputs(patch_name)
    model = get_model()
    height, width = inputs["mask"].shape
    folder = REPO / "output" / "offset_depth_sweep"
    folder.mkdir(parents=True, exist_ok=True)
    paths = []
    for offset in range(-4, 5):
        start = FLAT_DEPTH - (DEPTH - 1) // 2 + offset
        end = start + DEPTH
        if start < 0 or end > inputs["volume"].shape[0]:
            raise ValueError(f"Offset {offset:+d} selects invalid slices {start}:{end}")
        prediction = predict_tiles(
            C, model, inputs["volume"], inputs["mask"], inputs["coords"],
            (0, height), (0, width), C.data.d_start, f"offset_patch_d{offset:+d}", *inputs["norm"],
            surface_depth_map=inputs["surface_depth"],
            surface_confidence_map=inputs["surface_confidence"],
            surface_depth_offset=offset,
        )
        # raw probabilities for the depth-compositing cells below
        np.save(folder / f"{patch_name}_offset_{offset:+d}.npy", np.asarray(prediction, dtype=np.float32))
        projection = np.asarray(inputs["volume"][start:end]).max(axis=0)
        composite = np.clip(projection, 0, 255).astype(np.uint8) * inputs["mask"]
        shown_prediction = orient_display(prediction, patch_name)
        shown_composite = orient_display(composite, patch_name)
        panels = [[
            (f"{patch_name} | d{offset:+d} | slices {start}-{end - 1}",
             colorize(shown_prediction, "gray_r", (0.78, 0.86, 0.95)), cv2.INTER_NEAREST, None),
            (f"composite {start}-{end - 1}", gray_rgb(shown_composite), cv2.INTER_AREA, 9.362),
        ]]
        path = folder / f"{patch_name}_offset_{offset:+d}.png"
        save_panel_grid(panels, shown_composite.shape, str(path), gap_px=24)
        paths.append(path)
        print(f"Saved {path}")
        del prediction, projection, composite, shown_prediction, shown_composite, panels
    del inputs
    gc.collect()
    if C.device == "cuda":
        torch.cuda.empty_cache()
    return paths


offset_sweep_paths = sweep_offset_depth(STRAIGHT)


In [ ]:
# DEPTH COMPOSITES, straight_abf: combine the sweep's per-depth predictions with four compositing methods.
OFFSET_RANGE = (-3, 3)  # inclusive
TOPK_MEAN = 2           # mean of the k highest probabilities: 1 = max, len(range) = plain mean
ZSCORE_TOPK = 2         # strongest per-depth z-scores averaged per pixel
CORROB_K = 3            # a cell counts as corroborated when its k-th strongest depth is also high
CORROB_GAMMA = 0.5      # mid-tone lift for fully corroborated cells (1 = none; 0.5 maps 0.25 -> 0.5)
CORROB_SIGMA = 1.0      # spatial agreement, in prediction cells (one cell = 16 px)
CORROB_RAMP = (0.15, 0.45)  # k-th-depth support from no boost to full boost


def sweep_predictions(patch_name, offsets, inputs):
    """per-offset probability maps from the sweep cache; offsets missing from it are predicted and cached."""
    folder = REPO / "output" / "offset_depth_sweep"
    folder.mkdir(parents=True, exist_ok=True)
    height, width = inputs["mask"].shape
    maps = []
    for offset in offsets:
        path = folder / f"{patch_name}_offset_{offset:+d}.npy"
        if not path.is_file():
            print(f"predicting offset {offset:+d} (not in the sweep cache)")
            prediction = predict_tiles(
                C, get_model(), inputs["volume"], inputs["mask"], inputs["coords"],
                (0, height), (0, width), C.data.d_start, f"offset_patch_d{offset:+d}", *inputs["norm"],
                surface_depth_map=inputs["surface_depth"],
                surface_confidence_map=inputs["surface_confidence"],
                surface_depth_offset=offset,
            )
            np.save(path, np.asarray(prediction, dtype=np.float32))
        maps.append(np.load(path))
    return np.stack(maps)


def depth_composites(stack):
    """{method: (map, vmin, vmax)} for max, corroborated mid-boost, top-k mean and z-score top-k."""
    valid = np.isfinite(stack).all(axis=0)
    probability = np.clip(np.nan_to_num(stack, nan=0.5), 1e-4, 1 - 1e-4)
    ranked = np.sort(probability, axis=0)[::-1]
    top_mean = ranked[:min(TOPK_MEAN, len(ranked))].mean(axis=0)
    # corroboration: the k-th strongest depth, averaged over neighbouring cells (NaN-aware)
    k = min(CORROB_K, len(ranked))
    weight = cv2.GaussianBlur(valid.astype(np.float32), (0, 0), CORROB_SIGMA)
    support = cv2.GaussianBlur((ranked[k - 1] * valid).astype(np.float32), (0, 0), CORROB_SIGMA)
    support /= np.maximum(weight, 1e-6)
    agree = np.clip((support - CORROB_RAMP[0]) / (CORROB_RAMP[1] - CORROB_RAMP[0]), 0, 1)
    agree = agree * agree * (3 - 2 * agree)
    # lone single-depth spikes drop to the top-k mean; agreed cells get their mids lifted by the gamma curve
    corroborated = ranked[:k].mean(axis=0) ** (1 - (1 - CORROB_GAMMA) * agree)
    # each depth on its own noise scale (median/MAD of its logits), so a noisy depth cannot dominate
    logits = np.log(probability) - np.log1p(-probability)
    z = np.empty_like(logits)
    for index, depth_logits in enumerate(logits):
        values = depth_logits[valid]
        median = np.median(values)
        z[index] = (depth_logits - median) / (1.4826 * np.median(np.abs(values - median)) + 1e-6)
    z_top = np.sort(z, axis=0)[-min(ZSCORE_TOPK, len(z)):].mean(axis=0)
    nan = np.where(valid, 0.0, np.nan)
    return {
        "max": (ranked[0] + nan, 0.0, 1.0),
        "corroborated": (corroborated + nan, 0.0, 1.0),
        "topk_mean": (top_mean + nan, 0.0, 1.0),
        "zscore_top": (z_top + nan, 0.0, float(np.percentile(z_top[valid], 99.5))),
    }


def composite_offset_range(patch_name, offset_range=None):
    from utils.fast_figure import orient_display

    low, high = offset_range or OFFSET_RANGE
    if low > high:
        raise ValueError(f"Empty offset range {(low, high)}")
    offsets = list(range(low, high + 1))
    inputs = load_inputs(patch_name)
    first = FLAT_DEPTH - (DEPTH - 1) // 2 + low
    last = FLAT_DEPTH - (DEPTH - 1) // 2 + high + DEPTH
    if first < 0 or last > inputs["volume"].shape[0]:
        raise ValueError(f"Offsets {low:+d}..{high:+d} select invalid slices {first}:{last}")
    composites = depth_composites(sweep_predictions(patch_name, offsets, inputs))
    cell_hw = orient_display(inputs["mask"], patch_name).shape
    folder = REPO / "output" / "offset_depth_sweep" / "composites"
    folder.mkdir(parents=True, exist_ok=True)
    stem = f"{patch_name}_d{low:+d}_{high:+d}"
    labels = {"max": "max",
              "corroborated": f"corroborated mid-boost k={CORROB_K} gamma={CORROB_GAMMA:g}",
              "topk_mean": f"top-{TOPK_MEAN} mean", "zscore_top": f"z-score top-{ZSCORE_TOPK} (0..p99.5)"}
    panels = {}
    for method, (values, vmin, vmax) in composites.items():
        image = colorize(orient_display(values, patch_name), "gray_r", (0.78, 0.86, 0.95), vmin=vmin, vmax=vmax)
        panels[method] = (f"{labels[method]} | d{low:+d}..{high:+d}", image, cv2.INTER_AREA, None)
        path = folder / f"{stem}_{method}.png"
        save_panel_grid([[panels[method]]], cell_hw, str(path), gap_px=24)
        print(f"Saved {path}")
    comparison = folder / f"{stem}_compare.png"
    save_panel_grid([[panels["max"], panels["corroborated"]], [panels["topk_mean"], panels["zscore_top"]]],
                    cell_hw, str(comparison), gap_px=24)
    print(f"Saved {comparison}")
    with Image.open(comparison) as saved:
        display(saved.resize((max(1, saved.width // PREVIEW_DOWNSCALE), max(1, saved.height // PREVIEW_DOWNSCALE)),
                             Image.Resampling.LANCZOS))
    del inputs, composites, panels
    gc.collect()
    return comparison


composite_path = composite_offset_range(STRAIGHT)


In [ ]:
# Z-SCORE + CT 2x2, straight_abf: z-score top-k beside three copies of the CT composite (cached sweep predictions).
def zscore_ct_grid(patch_name, offset_range=None):
    from utils.fast_figure import orient_display

    low, high = offset_range or OFFSET_RANGE
    inputs = load_inputs(patch_name)
    first = FLAT_DEPTH - (DEPTH - 1) // 2 + low
    last = FLAT_DEPTH - (DEPTH - 1) // 2 + high + DEPTH
    if low > high or first < 0 or last > inputs["volume"].shape[0]:
        raise ValueError(f"Offsets {low:+d}..{high:+d} select invalid slices {first}:{last}")
    zscore, vmin, vmax = depth_composites(sweep_predictions(patch_name, list(range(low, high + 1)), inputs))["zscore_top"]
    projection = np.asarray(inputs["volume"][first:last]).max(axis=0)
    ct = orient_display(np.clip(projection, 0, 255).astype(np.uint8) * inputs["mask"], patch_name)
    z_panel = (f"z-score top-{ZSCORE_TOPK} (0..p99.5) | d{low:+d}..{high:+d}",
               colorize(orient_display(zscore, patch_name), "gray_r", (0.78, 0.86, 0.95), vmin=vmin, vmax=vmax),
               cv2.INTER_AREA, None)
    ct_panel = (f"CT max slices {first}-{last - 1}", gray_rgb(ct), cv2.INTER_AREA, 9.362)
    folder = REPO / "output" / "offset_depth_sweep" / "composites"
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"{patch_name}_d{low:+d}_{high:+d}_zscore_ct_2x2.png"
    save_panel_grid([[z_panel, ct_panel], [ct_panel, ct_panel]], ct.shape, str(path), gap_px=24)
    print(f"Saved {path}")
    with Image.open(path) as saved:
        display(saved.resize((max(1, saved.width // PREVIEW_DOWNSCALE), max(1, saved.height // PREVIEW_DOWNSCALE)),
                             Image.Resampling.LANCZOS))
    del inputs, zscore, projection, ct
    gc.collect()
    return path


zscore_grid_path = zscore_ct_grid(STRAIGHT)


In [ ]:
# OFFSET SWEEP, straight_abf_m28 (functions defined in the straight_abf cells above).
offset_sweep_paths_m28 = sweep_offset_depth(STRAIGHT_M28)


In [ ]:
# DEPTH COMPOSITES, straight_abf_m28.
composite_path_m28 = composite_offset_range(STRAIGHT_M28)


In [ ]:
# Z-SCORE + CT 2x2, straight_abf_m28.
zscore_grid_path_m28 = zscore_ct_grid(STRAIGHT_M28)


In [ ]:
# TRAINING EVAL FIGURES: the trainer's own eval_int path (TensorboardVisualizer.add_evaluation_figures with the
# checkpoint's full run config), for the three trained PHerc0846A patches. The only change is the bottom row,
# which shows the input composite twice instead of the raw 1.1 / 2.4 um inklabels.
from utils.fast_figure import orient_display

TRAIN_EVAL_IDS = (20260927134931, 20261012073246)
TRAIN_EVAL_DIR = OUT_DIR / "training_eval_figs"


def training_eval_config():
    """the checkpoint's saved run config (data, loader, training, model), exactly as the trainer used it."""
    config = Config()
    for section in ("data", "dl", "tra", "model"):
        for name, value in RUN_CONFIG[section].items():
            if section == "data" and name == "scrolls":
                config.data.scrolls = [ScrollConfig(**item) for item in value]
            elif name != "probe_rois" and hasattr(getattr(config, section), name):
                setattr(getattr(config, section), name, value)
    config.device = C.device
    config.data.zarr_path = str(ZARR_DIR)
    config.model.compile_model = False
    return config


class _NullWriter:
    """swallows the scalars and TensorBoard figures the eval path logs."""

    def __getattr__(self, name):
        return lambda *args, **kwargs: None


class CompositeEvalVisualizer(TensorboardVisualizer):
    """the per-scroll training visualizer without TensorBoard or test fragments; bottom row = composite twice."""

    def __init__(self, config, scroll_id, out_dir):
        self.c, self.mode, self.scroll1_id = config, "train", int(scroll_id)
        self.eval_enabled, self._load_test_frags = True, False
        self.probe_log_interval = max(1, int(getattr(config.tra, "probe_int", 5)))
        self._init_training_assets()
        self.log_path, self.writer = str(out_dir), _NullWriter()

    def _eval_composite(self):
        """max projection of the slices the eval window reads, over the eval extent (manual-split full range)."""
        from utils.visualizer import _configured_surface_window_offset

        (y0, y1), (x0, x1) = self.full_y_range, self.full_x_range
        depth = np.asarray(self.surface_depth_map[y0:y1, x0:x1])
        centre = int(np.median(depth[depth != 255]))
        start = centre - (self.c.data.depth - 1) // 2 + _configured_surface_window_offset(self.c, self.scroll1_id)
        end = start + self.c.data.depth
        suffix = (self.c.data.zarr_suffix or {}).get(str(self.scroll1_id), "")
        volume = zarr.open(str(ZARR_DIR / f"{self.scroll1_id}{suffix}.zarr"), mode="r")
        projection = np.asarray(volume[start:end, y0:y1, x0:x1]).max(axis=0)
        composite = np.clip(projection, 0, 255).astype(np.uint8) * (np.asarray(self.mask[y0:y1, x0:x1]) > 0)
        return composite, start, end

    def _create_eval_figure_2x2(self, reg_pred, tta_pred, label_binary, raw_1_1, raw_2_4, *args, **kwargs):
        from utils.visualizer import _configured_surface_window_offset

        fig = super()._create_eval_figure_2x2(reg_pred, tta_pred, label_binary, None, None, *args, **kwargs)
        composite, start, end = self._eval_composite()
        composite = orient_display(composite, self.scroll1_id)
        for ax in fig.axes[2:4]:
            ax.imshow(composite, cmap="gray", vmin=0, vmax=255, aspect="equal")
            ax.set_title(f"composite {start}-{end - 1}", fontsize=8)
            ax.axis("off")
        offsets, method = depth_ensemble_utils.resolve(depth_ensemble, ensemble_method)
        base = _configured_surface_window_offset(self.c, self.scroll1_id)
        ensemble_label = depth_ensemble_utils.describe([base + o for o in offsets], method)
        if ensemble_label:
            fig.suptitle(f"scroll {self.scroll1_id} | {ensemble_label}", fontsize=9)
        return fig


def training_eval_figure(scroll_id, config):
    """render one scroll's final-epoch eval figure; returns the saved full-size PNG."""
    import utils.visualizer as visualizer_module

    offsets, method = depth_ensemble_utils.resolve(depth_ensemble, ensemble_method)
    single_depth = visualizer_module.predict_tiles
    visualizer = CompositeEvalVisualizer(config, scroll_id, TRAIN_EVAL_DIR)
    try:
        # the trainer's eval path calls the module-level predict_tiles; ensemble it for this figure only
        visualizer_module.predict_tiles = depth_ensemble_utils.ensemble_predict(single_depth, offsets, method)
        visualizer.add_evaluation_figures(int(config.tra.n_epochs) - 1, get_model())
    finally:
        visualizer_module.predict_tiles = single_depth
        del visualizer
        gc.collect()
        if C.device == "cuda":
            torch.cuda.empty_cache()
    return TRAIN_EVAL_DIR / "eval_figs" / f"eval_s{scroll_id}_ep{int(config.tra.n_epochs):02d}.png"


In [ ]:
# Render the three training eval figures, save them full size, and preview each reduced.
train_eval_config = training_eval_config()
train_eval_paths = []
for scroll_id in TRAIN_EVAL_IDS:
    path = training_eval_figure(scroll_id, train_eval_config)
    train_eval_paths.append(path)
    print(f"Saved {path}")
    with Image.open(path) as saved:
        display(saved.resize((max(1, saved.width // 4), max(1, saved.height // 4)), Image.Resampling.LANCZOS))
